# Creation of the BGC climatology used for initialization and/or boundary conditions as well as restoring
- interpolates the GLODAPv2 data to the ocean physics input data
- for some variables, negative values are set to zeros.
- for variables expressed in per kg are expressed in per m3 to match with the model units.

In [51]:
import os
import xarray as xr
import numpy as np
import gsw  # Gibbs SeaWater library for accurate density

# --- 1. CONFIGURATION ---
ref_exp = "jcope_nwpac025"
ref_file = "interp_T_01JAN1990.nc"
ref_var = "tm"
glodap_dir = "../climatology/GLODAPv2.2016b.MappedProduct"

# Complete list of GLODAP variables to process
glodap_vars = [
    "Cant", "NO3", "OmegaA", "OmegaC", "oxygen", 
    "pHts25p0", "pHtsinsitutp", "PI_TCO2", "PO4", 
    "salinity", "silicate", "TAlk", "TCO2", "temperature"
]

# Variables that are in µmol/kg and need density conversion to mmol/m^3
vars_to_convert = [
    "Cant", "NO3", "oxygen", "PI_TCO2", "PO4", 
    "silicate", "TAlk", "TCO2"
]

# Variables that are physically strictly positive (concentrations, salinity, saturation states)
# Temperature and pH are excluded from this list so they are NOT clipped to 0
vars_strictly_positive = [
    "Cant", "NO3", "OmegaA", "OmegaC", "oxygen", 
    "PI_TCO2", "PO4", "salinity", "silicate", "TAlk", "TCO2"
]

# --- 2. LOAD REFERENCE GRID & HELPERS ---
ref_da = xr.open_dataset(f"../input/{ref_exp}/{ref_file}")[ref_var]

def standardize_glodap_grid(ds, ref_grid):
    """Aligns GLODAP coordinates and dimensions with the model grid."""
    if "depth_surface" in ds.dims:
        ds = ds.swap_dims({"depth_surface": "Depth"}).rename({"Depth": "depth"})
    
    # Adjust Longitude convention
    if ref_grid["lon"].min() < 0:
        ds.coords['lon'] = (ds.coords['lon'] + 180) % 360 - 180 
    else:
        ds.coords['lon'] = ds.coords['lon'] % 360
        
    return ds.sortby("lon")


# --- 3. CALCULATE IN-SITU DENSITY ---
print("Calculating in-situ density from GLODAP Temperature and Salinity...")
# We must load T and S first to calculate the density field used for converting the others
ds_temp = xr.open_dataset(f"{glodap_dir}/GLODAPv2.2016b.temperature.nc")
ds_salt = xr.open_dataset(f"{glodap_dir}/GLODAPv2.2016b.salinity.nc")

ds_temp_std = standardize_glodap_grid(ds_temp, ref_da)["temperature"]
ds_salt_std = standardize_glodap_grid(ds_salt, ref_da)["salinity"]

# TEOS-10 Density calculation
p = ds_temp_std.depth # Pressure approx 1 dbar per meter
SA = gsw.SA_from_SP(ds_salt_std, p, ds_salt_std.lon, ds_salt_std.lat)
CT = gsw.CT_from_t(SA, ds_temp_std, p)
density = gsw.rho(SA, CT, p)  # kg / m^3


# --- 4. BUILD THE MASTER GLODAP DATASET ---
ds_glodap = xr.Dataset()

for var in glodap_vars:
    print(f"Loading {var}...")
    
    # Load and standardize
    ds_var = xr.open_dataset(f"{glodap_dir}/GLODAPv2.2016b.{var}.nc")
    da = standardize_glodap_grid(ds_var, ref_da)[var]
    
    # A. Density Conversion (umol/kg -> mmol/m^3)
    if var in vars_to_convert:
        # C_vol = C_mass * (density / 1000.0)
        da = da * (density / 1000.0)
        da.attrs['units'] = 'mmol/m^3'
    
    # B. Clip negative artifacts from raw data (for concentrations only)
    if var in vars_strictly_positive:
        da = da.clip(min=0.0)
        
    ds_glodap[var] = da


# --- 5. FLOOD, REGRID, AND MASK ---
print("\nExtrapolating NaNs (Flooding)...")
# Flood horizontally to push ocean data into land gaps (so interpolation doesn't grab NaNs)
ds_filled = ds_glodap.interpolate_na(dim='lon', method='nearest', fill_value="extrapolate")
ds_filled = ds_filled.interpolate_na(dim='lat', method='nearest', fill_value="extrapolate")

print("Interpolating to Model Grid...")
# Regrid all 14 variables to the target grid simultaneously
ds_regridded = ds_filled.interp_like(ref_da.isel(time=0))

print("Filling vertical gaps...")
ds_regridded = ds_regridded.ffill(dim='depth')

# Interpolation math can sometimes create tiny negative numbers, so we re-clip the required variables
for var in vars_strictly_positive:
    ds_regridded[var] = ds_regridded[var].clip(min=0.0)

print("Applying Target Land Mask...")
target_mask = ref_da.isel(time=0).notnull()
ds_final = ds_regridded.where(target_mask)


# --- 6. SAVE TO DISK ---
out_dir = f"../climatology/{ref_exp}"
os.makedirs(out_dir, exist_ok=True)
out_file = f"{out_dir}/GLODAPv2.2016b.ALL_{ref_exp}.nc"

print(f"\nSaving merged climatology to: {out_file}")
# Compress the file (14 variables in 3D will be large without zlib)
comp = dict(zlib=True, complevel=5)
encoding = {var: comp for var in ds_final.data_vars}
ds_final.to_netcdf(out_file, encoding=encoding)

print("Done! Climatology successfully generated.")

Calculating in-situ density from GLODAP Temperature and Salinity...
Loading Cant...
Loading NO3...
Loading OmegaA...
Loading OmegaC...
Loading oxygen...
Loading pHts25p0...
Loading pHtsinsitutp...
Loading PI_TCO2...
Loading PO4...
Loading salinity...
Loading silicate...
Loading TAlk...
Loading TCO2...
Loading temperature...

Extrapolating NaNs (Flooding)...
Interpolating to Model Grid...
Filling vertical gaps...
Applying Target Land Mask...

Saving merged climatology to: ../climatology/jcope_nwpac025/GLODAPv2.2016b.ALL_jcope_nwpac025.nc
Done! Climatology successfully generated.
